# World Cup 2022 Prediction Report

Scope: summarize simulated standings and knockout path, compare key predictions to actual outcomes, and highlight improvement opportunities.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except Exception:
    sns = None

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
if PROJECT_ROOT.name != 'Project-3-WorldCup-2022-Predictions':
    PROJECT_ROOT = Path.cwd().resolve()

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

fixtures = pd.read_csv(PROCESSED_DIR / 'worldcup_2022_fixtures.csv', parse_dates=['date'])
group_preds = pd.read_csv(OUTPUTS_DIR / 'group_stage_predictions.csv')
knockout_preds = pd.read_csv(OUTPUTS_DIR / 'knockout_predictions.csv')

with open(OUTPUTS_DIR / 'tournament_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

group_preds.head(), knockout_preds.head(), summary

In [ ]:
def build_group_standings_from_predictions(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group_name, gdf in pred_df.groupby('group'):
        teams = sorted(set(gdf['home_team']).union(set(gdf['away_team'])))
        table = {t: {'group': group_name, 'team': t, 'points': 0, 'gf': 0, 'ga': 0, 'gd': 0} for t in teams}

        for r in gdf.itertuples(index=False):
            h, a = r.home_team, r.away_team
            hg, ag = int(r.pred_home_goals), int(r.pred_away_goals)

            table[h]['gf'] += hg
            table[h]['ga'] += ag
            table[a]['gf'] += ag
            table[a]['ga'] += hg
            table[h]['gd'] += hg - ag
            table[a]['gd'] += ag - hg

            if hg > ag:
                table[h]['points'] += 3
            elif hg < ag:
                table[a]['points'] += 3
            else:
                table[h]['points'] += 1
                table[a]['points'] += 1

        group_table = pd.DataFrame(table.values()).sort_values(['group', 'points', 'gd', 'gf', 'team'], ascending=[True, False, False, False, True])
        group_table['rank'] = group_table.groupby('group').cumcount() + 1
        rows.append(group_table)

    return pd.concat(rows, ignore_index=True)

pred_group_table = build_group_standings_from_predictions(group_preds)
pred_group_table

In [ ]:
for g in sorted(pred_group_table['group'].dropna().unique()):
    print(f'Group {g}')
    display(pred_group_table[pred_group_table['group'] == g][['rank', 'team', 'points', 'gf', 'ga', 'gd']])

In [ ]:
display(knockout_preds[['stage', 'match_label', 'home_team', 'away_team', 'pred_home_goals', 'pred_away_goals', 'winner']])

In [ ]:
actual_final = fixtures[fixtures['stage'] == 'Final'].iloc[0]
actual_champion = actual_final['home_team'] if actual_final['actual_home_goals'] > actual_final['actual_away_goals'] else actual_final['away_team']

print('Predicted champion:', summary.get('champion'))
print('Actual champion:', actual_champion)
print('Champion match:', summary.get('champion') == actual_champion)

In [ ]:
actual_group = fixtures[fixtures['stage'] == 'Group'].copy()
actual_group['actual_result'] = np.select(
    [actual_group['actual_home_goals'] > actual_group['actual_away_goals'], actual_group['actual_home_goals'] < actual_group['actual_away_goals']],
    ['H', 'A'],
    default='D'
)

group_eval = group_preds.merge(actual_group[['match_no', 'actual_result']], on='match_no', how='left')
group_accuracy = (group_eval['pred_result'] == group_eval['actual_result']).mean()

actual_knockout = fixtures[fixtures['stage'].isin(['R16', 'QF', 'SF', 'Final'])].copy()
actual_knockout['actual_result'] = np.select(
    [actual_knockout['actual_home_goals'] > actual_knockout['actual_away_goals'], actual_knockout['actual_home_goals'] < actual_knockout['actual_away_goals']],
    ['H', 'A'],
    default='D'
)

knock_eval = knockout_preds.copy()
knock_eval['key'] = knock_eval['home_team'] + ' vs ' + knock_eval['away_team']
actual_knockout['key'] = actual_knockout['home_team'] + ' vs ' + actual_knockout['away_team']
knock_eval = knock_eval.merge(actual_knockout[['stage', 'key', 'actual_result']], on=['stage', 'key'], how='left')
knockout_accuracy = (knock_eval['pred_result'] == knock_eval['actual_result']).mean()

print('Group-stage outcome accuracy:', round(float(group_accuracy), 3))
print('Knockout-stage outcome accuracy:', round(float(knockout_accuracy), 3))

In [ ]:
conf_table = pd.crosstab(group_eval['actual_result'], group_eval['pred_result'], rownames=['actual'], colnames=['predicted'], dropna=False)
conf_table

In [ ]:
finalists = knockout_preds[knockout_preds['stage'] == 'Final'][['home_team', 'away_team', 'p_home_win', 'p_away_win']].iloc[0]
chart_df = pd.DataFrame({
    'team': [finalists['home_team'], finalists['away_team']],
    'win_probability': [float(finalists['p_home_win']), float(finalists['p_away_win'])]
})

plt.figure(figsize=(8, 4))
if sns is not None:
    sns.barplot(data=chart_df, x='team', y='win_probability', palette='Blues_d')
else:
    plt.bar(chart_df['team'], chart_df['win_probability'])

plt.title('Predicted Final Win Probabilities')
plt.ylim(0, 1)
plt.ylabel('Probability')
plt.xlabel('Team')
plt.show()

## Interpretation And Next Improvements
- The report compares the simulated bracket with actual results, but match-level probabilities can still be noisy with compact features.
- Strong next steps: add richer context features (rest days, opponent strength trajectory, confederation interaction effects), calibrate probabilities, and run many tournament simulations to estimate champion distributions instead of a single run.